In [5]:
using LinearAlgebra
include("pde_model.jl")

f_fun!

In [38]:
n = 64
d = 2
N = n^d

4096

In [36]:
FType=Float64

Float64

In [3]:
include("main_2.jl")

compute_BC_corr

In [9]:
ntuple(i -> n, d)

(64, 64)

In [49]:
buff = Vector{Float16}(undef, N)
buff .= 1;

In [50]:
grid = get_grid_points_as_1d_vect(n,d, FType);
buff = Vector{Float64}(undef, N)
buff .= 1
buff .= f_fun!(buff, grid, 0.0)
U = reshape(buff, ntuple(i -> n, d)) 

64×64 Matrix{Float64}:
  0.0796676   0.156367   0.22724   …  -0.22724   -0.156367  -0.0796676
  0.156367    0.306908   0.446013     -0.446013  -0.306908  -0.156367
  0.22724     0.446013   0.648168     -0.648168  -0.446013  -0.22724
  0.289646    0.5685     0.826172     -0.826172  -0.5685    -0.289646
  0.34126     0.669805   0.973393     -0.973393  -0.669805  -0.34126
  0.380159    0.746153   1.08435   …  -1.08435   -0.746153  -0.380159
  0.404893    0.7947     1.1549       -1.1549    -0.7947    -0.404893
  0.414541    0.813636   1.18242      -1.18242   -0.813636  -0.414541
  0.408743    0.802257   1.16588      -1.16588   -0.802257  -0.408743
  0.387716    0.760985   1.1059       -1.1059    -0.760985  -0.387716
  0.352242    0.69136    1.00472   …  -1.00472   -0.69136   -0.352242
  0.303644    0.595974   0.866099     -0.866099  -0.595974  -0.303644
  0.243732    0.478383   0.695209     -0.695209  -0.478383  -0.243732
  ⋮                                ⋱                        
 -0.303

In [15]:
using BenchmarkTools

In [24]:
n = 64
d = 3
stride = ntuple(k -> n^(k-1), d)
#@btime for idx=1:n^d; I = ntuple(k -> ((div(idx-1, stride[k]) % n) + 1), d); end
#@btime for idx=1:n^d; I = idx % n +1; end
idx = 199
I = ntuple(k -> ((div(idx-1, stride[k]) % n) + 1), d)

(7, 4, 1)

In [ ]:

for k in 1:d
    ((div(idx-1, stride[k]) % n) + 1)
end

741

In [4]:
#@allocated sparse(I, N, N) - 0.5 * L
nn = 4
NN = nn^d
sparse(I, NN, NN) - 0.5 * get_laplace_sparse_matrix(nn, d)

16×16 SparseMatrixCSC{Float64, Int64} with 64 stored entries:
⎡⠻⣦⠑⢄⠀⠀⠀⠀⎤
⎢⠑⢄⠻⣦⠑⢄⠀⠀⎥
⎢⠀⠀⠑⢄⠻⣦⠑⢄⎥
⎣⠀⠀⠀⠀⠑⢄⠻⣦⎦

In [5]:
L = get_laplace_sparse_matrix(n,d);

In [6]:
using TimerOutputs
const to = TimerOutput()
@timeit to "A_impl" A_impl = sparse(I, N, N) - 0.44234233 * L;
show(to)

────────────────────────────────────────────────────────────────────
                           Time                    Allocations      
                  ───────────────────────   ────────────────────────
Tot / % measured:      251ms /   0.1%           63.5MiB /   1.3%    

Section   ncalls     time    %tot     avg     alloc    %tot      avg
────────────────────────────────────────────────────────────────────
A_impl         1    154μs  100.0%   154μs    829KiB  100.0%   829KiB
────────────────────────────────────────────────────────────────────

---

In [35]:
@allocated 1.5 * get_laplace_sparse_matrix(n,d)

1251088

In [36]:
using TimerOutputs
const to = TimerOutput()
@timeit to "L" L = 1.5 * get_laplace_sparse_matrix(n,d);
show(to)

────────────────────────────────────────────────────────────────────
                           Time                    Allocations      
                  ───────────────────────   ────────────────────────
Tot / % measured:     1.25ms /  23.3%           1.21MiB /  98.3%    

Section   ncalls     time    %tot     avg     alloc    %tot      avg
────────────────────────────────────────────────────────────────────
L              1    291μs  100.0%   291μs   1.19MiB  100.0%  1.19MiB
────────────────────────────────────────────────────────────────────

---

In [3]:
function grid_coords(n::Int, d::Int)
    N = n^d
    h = 1/(n+1)
    coords = Matrix{Float64}(undef, N, d)

    for k in 1:d
        inner = n^(k-1)
        period = n^k
        for i in 1:N
            coords[i, k] = (i-1) % period ÷ inner + 1
        end
    end

    return h * coords
end

grid_coords (generic function with 1 method)

In [4]:
# -- warm-up call --
# first run of the function compiles it
# run it first with some small args to trigger the compilation
# then run it again with the correct arguments
grid_coords(3,2)

9×2 Matrix{Float64}:
 0.25  0.25
 0.5   0.25
 0.75  0.25
 0.25  0.5
 0.5   0.5
 0.75  0.5
 0.25  0.75
 0.5   0.75
 0.75  0.75

In [5]:
@allocated grid_coords(n,d)

131248

In [6]:
using TimerOutputs
const to = TimerOutput()
@timeit to "grid" grid = grid_coords(n,d);
show(to)

────────────────────────────────────────────────────────────────────
                           Time                    Allocations      
                  ───────────────────────   ────────────────────────
Tot / % measured:      338ms /   0.0%           63.4MiB /   0.2%    

Section   ncalls     time    %tot     avg     alloc    %tot      avg
────────────────────────────────────────────────────────────────────
grid           1   52.2μs  100.0%  52.2μs    129KiB  100.0%   129KiB
────────────────────────────────────────────────────────────────────

---

In [7]:
u_analytic_fun(grid[1:4,:], 0.0)

4-element Vector{Float64}:
 0.03691267552111176
 0.07244999406459336
 0.10528784410682773
 0.1342026956395355

In [8]:
@allocated u_analytic_fun(grid, 0.0)

33000

In [9]:
using TimerOutputs
const to = TimerOutput()
@timeit to "evaled" u = u_analytic_fun(grid, 0.0)
show(to)

────────────────────────────────────────────────────────────────────
                           Time                    Allocations      
                  ───────────────────────   ────────────────────────
Tot / % measured:      908μs /   7.5%           53.1KiB /  61.2%    

Section   ncalls     time    %tot     avg     alloc    %tot      avg
────────────────────────────────────────────────────────────────────
evaled         1   67.7μs  100.0%  67.7μs   32.5KiB  100.0%  32.5KiB
────────────────────────────────────────────────────────────────────